In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split

DATA_PATH = Path("/content/housing.csv")
df = pd.read_csv(DATA_PATH)

TARGET = "median_house_value"
LOCATION_FEATURES = ["longitude", "latitude"]
BASELINE_FEATURES = ["median_income"]
NON_LOCATION_FEATURES = [
    "housing_median_age",
    "total_rooms",
    "total_bedrooms",
    "population",
    "households",
    "median_income",
    "ocean_proximity",
]
LOCATION_FEATURE_SET = NON_LOCATION_FEATURES + LOCATION_FEATURES
CATEGORICAL_FEATURES = ["ocean_proximity"]

required_columns = set([TARGET] + LOCATION_FEATURE_SET)
missing_columns = sorted(required_columns.difference(df.columns))
if missing_columns:
    raise ValueError(f"Required columns missing from loaded data: {missing_columns}")

print(f"Loaded: {DATA_PATH.name} | rows={len(df):,} | columns={df.shape[1]}")
print("Column types:")
print(df.dtypes.to_string())

Loaded: housing.csv | rows=20,640 | columns=10
Column types:
longitude             float64
latitude              float64
housing_median_age      int64
total_rooms             int64
total_bedrooms        float64
population              int64
households              int64
median_income         float64
median_house_value      int64
ocean_proximity        object


In [3]:
SEEDS = (17, 29, 41, 53, 67)
TEST_FRACTION = 0.20
SPATIAL_CELL_DEGREES = 1.0
RF_PARAMS = {
    "n_estimators": 300,
    "min_samples_leaf": 2,
    "random_state": 17,
    "n_jobs": -1,
}

def make_random_splits(n_rows, seeds=SEEDS, test_size=TEST_FRACTION):
    """Return repeat, train-position, test-position tuples for row-level holdouts."""
    positions = np.arange(n_rows)
    return [
        (repeat, *train_test_split(
            positions, test_size=test_size, random_state=seed, shuffle=True
        ))
        for repeat, seed in enumerate(seeds, start=1)
    ]

def make_spatial_groups(frame, cell_degrees=SPATIAL_CELL_DEGREES):
    """Assign each row to a fixed-size longitude/latitude grid cell."""
    coordinates = frame[LOCATION_FEATURES]
    if coordinates.isna().any().any():
        raise ValueError("Spatial holdouts require non-missing longitude and latitude.")
    lon_cell = np.floor(coordinates["longitude"].to_numpy() / cell_degrees).astype("int64")
    lat_cell = np.floor(coordinates["latitude"].to_numpy() / cell_degrees).astype("int64")
    return np.char.add(np.char.add(lon_cell.astype(str), "_"), lat_cell.astype(str))

def make_spatial_splits(groups, seeds=SEEDS, test_size=TEST_FRACTION):
    """Return repeat, train-position, test-position tuples with whole cells held out."""
    positions = np.arange(len(groups))
    splits = []
    for repeat, seed in enumerate(seeds, start=1):
        splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
        train_pos, test_pos = next(splitter.split(positions, groups=groups))
        splits.append((repeat, train_pos, test_pos))
    return splits

def regression_metrics(y_true, y_pred):
    """Return MAE and RMSE in the units of the target."""
    errors = np.asarray(y_true) - np.asarray(y_pred)
    return {
        "MAE": float(np.mean(np.abs(errors))),
        "RMSE": float(np.sqrt(np.mean(errors ** 2))),
    }

spatial_groups = make_spatial_groups(df)
random_splits = make_random_splits(len(df))
spatial_splits = make_spatial_splits(spatial_groups)

split_rows = []
for design, splits in [("random", random_splits), ("spatial", spatial_splits)]:
    for repeat, train_pos, test_pos in splits:
        split_rows.append({
            "design": design,
            "repeat": repeat,
            "train_rows": len(train_pos),
            "test_rows": len(test_pos),
            "test_cells": int(pd.Series(spatial_groups[test_pos]).nunique()),
        })
        assert not np.intersect1d(train_pos, test_pos).size
        if design == "spatial":
            assert not set(spatial_groups[train_pos]).intersection(spatial_groups[test_pos])

print(f"Spatial grouping: {SPATIAL_CELL_DEGREES:g}-degree cells | unique cells={len(np.unique(spatial_groups)):,}")
print("Split sample sizes (test-cell count is descriptive for both designs):")
display(pd.DataFrame(split_rows).to_string(index=False))
print("Models: income-only LinearRegression; RandomForestRegressor on non-location features; RandomForestRegressor on non-location + coordinates.")
print(f"Random Forest settings: {RF_PARAMS}")

Spatial grouping: 1-degree cells | unique cells=55
Split sample sizes (test-cell count is descriptive for both designs):


' design  repeat  train_rows  test_rows  test_cells\n random       1       16512       4128          49\n random       2       16512       4128          49\n random       3       16512       4128          49\n random       4       16512       4128          51\n random       5       16512       4128          48\nspatial       1       18763       1877          11\nspatial       2       14039       6601          11\nspatial       3       18234       2406          11\nspatial       4       15098       5542          11\nspatial       5       18708       1932          11'

Models: income-only LinearRegression; RandomForestRegressor on non-location features; RandomForestRegressor on non-location + coordinates.
Random Forest settings: {'n_estimators': 300, 'min_samples_leaf': 2, 'random_state': 17, 'n_jobs': -1}


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Summarize missing values for the target and every candidate predictor.
checked_columns = [TARGET] + LOCATION_FEATURE_SET
missing_summary = (
    df[checked_columns]
    .isna()
    .sum()
    .rename("missing_rows")
    .to_frame()
    .assign(missing_pct=lambda table: 100 * table["missing_rows"] / len(df))
)
print(f"Rows available: {len(df):,} | rows with missing target: {int(df[TARGET].isna().sum()):,}")
print("Missingness in target and candidate predictors:")
print(missing_summary.to_string(formatters={"missing_pct": "{:.2f}%".format}))

if df[TARGET].isna().any():
    raise ValueError("The target contains missing values; define a target-row exclusion rule before fitting.")

# Report the observed target ceiling and its frequency, rather than assuming it from metadata.
observed_target_max = df[TARGET].max()
at_observed_max = int(df[TARGET].eq(observed_target_max).sum())
print(
    f"Target range: {df[TARGET].min():,.0f} to {observed_target_max:,.0f}; "
    f"rows at observed maximum: {at_observed_max:,} ({at_observed_max / len(df):.2%})"
)
print("Most frequent target values:")
print(df[TARGET].value_counts().head(5).rename_axis(TARGET).to_string())

print("Categorical values (including missing, if any):")
print(df["ocean_proximity"].value_counts(dropna=False).rename_axis("ocean_proximity").to_string())

Rows available: 20,640 | rows with missing target: 0
Missingness in target and candidate predictors:
                    missing_rows missing_pct
median_house_value             0       0.00%
housing_median_age             0       0.00%
total_rooms                    0       0.00%
total_bedrooms               207       1.00%
population                     0       0.00%
households                     0       0.00%
median_income                  0       0.00%
ocean_proximity                0       0.00%
longitude                      0       0.00%
latitude                       0       0.00%
Target range: 14,999 to 500,001; rows at observed maximum: 965 (4.68%)
Most frequent target values:
median_house_value
500001    965
137500    122
162500    117
112500    103
187500     93
Categorical values (including missing, if any):
ocean_proximity
<1H OCEAN     9136
INLAND        6551
NEAR OCEAN    2658
NEAR BAY      2290
ISLAND           5


In [5]:
MODEL_FEATURES = {
    "Income-only LinearRegression": BASELINE_FEATURES,
    "Random Forest (non-location)": NON_LOCATION_FEATURES,
    "Random Forest (+ coordinates)": LOCATION_FEATURE_SET,
}

def make_preprocessor(features):
    """Create an unfitted transformer for the requested predictor columns."""
    categorical = [feature for feature in features if feature in CATEGORICAL_FEATURES]
    numeric = [feature for feature in features if feature not in CATEGORICAL_FEATURES]
    transformers = []

    if numeric:
        numeric_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ])
        transformers.append(("numeric", numeric_pipeline, numeric))

    if categorical:
        categorical_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ])
        transformers.append(("categorical", categorical_pipeline, categorical))

    return ColumnTransformer(transformers=transformers, remainder="drop")

def make_model_pipeline(model_name):
    """Return a fresh, unfitted preprocessing-and-estimator pipeline."""
    features = MODEL_FEATURES[model_name]
    if model_name == "Income-only LinearRegression":
        estimator = LinearRegression()
    else:
        estimator = RandomForestRegressor(**RF_PARAMS)

    return Pipeline([
        ("preprocess", make_preprocessor(features)),
        ("model", estimator),
    ])

pipeline_summary = []
for model_name, features in MODEL_FEATURES.items():
    pipeline = make_model_pipeline(model_name)
    pipeline_summary.append({
        "model": model_name,
        "predictor_count": len(features),
        "numeric_predictors": ", ".join(
            feature for feature in features if feature not in CATEGORICAL_FEATURES
        ),
        "categorical_predictors": ", ".join(
            feature for feature in features if feature in CATEGORICAL_FEATURES
        ) or "—",
        "pipeline_steps": " -> ".join(name for name, _ in pipeline.steps),
    })

print("Unfitted pipeline specifications (fit separately on each training split):")
print(pd.DataFrame(pipeline_summary).to_string(index=False))
print("Preprocessing: numeric median imputation; categorical most-frequent imputation + one-hot encoding; unseen categories ignored.")

Unfitted pipeline specifications (fit separately on each training split):
                        model  predictor_count                                                                                          numeric_predictors categorical_predictors      pipeline_steps
 Income-only LinearRegression                1                                                                                               median_income                      — preprocess -> model
 Random Forest (non-location)                7                      housing_median_age, total_rooms, total_bedrooms, population, households, median_income        ocean_proximity preprocess -> model
Random Forest (+ coordinates)                9 housing_median_age, total_rooms, total_bedrooms, population, households, median_income, longitude, latitude        ocean_proximity preprocess -> model
Preprocessing: numeric median imputation; categorical most-frequent imputation + one-hot encoding; unseen categories ignored.
